# 🏆 SOTA v21 Hybrid Stacking-Blending Calibrated Ensemble (Leave-One-Subject-Out)

This notebook implements the complete pipeline for **SOTA v18, v20, and v21 Leave-One-Subject-Out (LOSO) Calibrated Ensemble** models.
This pipeline was a monumental milestone (ID 26 ~ 48 in `submission_log.csv`) that drove our local CV down to **`0.60081`**.

### Landmark Features:
1. **Leave-One-Subject-Out Cross-Validation (LOSO)**: Out-of-fold generalization mapped strictly across distinct subject partitions.
2. **Grid-Search Temperature Probability Calibration**: Applying global and subject-wise beta smoothing to avoid class clipping issues.
3. **Hybrid Meta-Stacking Decision**: Automatically evaluating both SLSQP Blending and Logistic Regression Stacking per target to map the best generalization score.

In [ ]:
import os
import sys
import warnings
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.metrics import log_loss

warnings.filterwarnings('ignore')

project_root = Path(os.getcwd()).resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.data.data_loader import DataLoader
from src.data.make_dataset import build_training_dataset
from src.features.build_features import engineer_features
from src.features.transformers import LeakFreeSubjectTargetEncoder, LeakFreeSubjectZScaler
from src.utils.logger import get_logger
from src.utils.validation import make_subject_hole_folds

logger = get_logger(__name__)
print("✅ SOTA v18-v21 LOSO Ensemble libraries successfully loaded!")

In [ ]:
print("🔄 Initializing SOTA-v21 Base Data Loader & Feature Engineering...")
data_loader = DataLoader(data_dir="data")
metrics_df = data_loader.load_metrics()
sample_sub_df = data_loader.load_sample_submission()

processed_train_path = Path("data/processed/df_train_raw.parquet")
processed_test_path = Path("data/processed/df_test_raw.parquet")

if processed_train_path.exists() and processed_test_path.exists():
    print("Loading cached raw training/testing datasets...")
    df_train_raw = pd.read_parquet(processed_train_path)
    df_test_raw = pd.read_parquet(processed_test_path)
else:
    print("No cached datasets found. Rebuilding...")
    sensor_data_dict = data_loader.load_sensor_data()
    df_train_raw = build_training_dataset(metrics_df, sensor_data_dict)
    df_test_raw = build_training_dataset(sample_sub_df, sensor_data_dict)

df_train_raw['is_test'] = 0
df_test_raw['is_test'] = 1
df_all_raw = pd.concat([df_train_raw, df_test_raw], ignore_index=True)

df_all = engineer_features(df_all_raw)

df_train = df_all[df_all['is_test'] == 0].drop(columns=['is_test']).copy()
df_test = df_all[df_all['is_test'] == 1].drop(columns=['is_test']).copy()

id_col = "subject_id" if "subject_id" in df_train.columns else "user_id"
targets = ["Q1", "Q2", "Q3", "S1", "S2", "S3", "S4"]

print(f"Data loaded! Train shape: {df_train.shape}, Test shape: {df_test.shape}")

In [ ]:
print("🔄 Setting up Leave-One-Subject-Out (LOSO) K-Fold Split & Base Models...")
n_splits = 5
df_train = df_train.sort_values(by=[id_col, 'date']).reset_index(drop=True)
df_test = df_test.sort_values(by=[id_col, 'date']).reset_index(drop=True)

cv_folds = make_subject_hole_folds(df_train, n_folds=n_splits)

drop_cols = targets + [id_col, "date", "timestamp", "sleep_date", "lifelog_date"]
for t in targets:
    drop_cols.extend([f"{t}_zscore", f"{t}_is_missing"])

feature_cols = [c for c in df_train.columns if c not in drop_cols]
numeric_cols = df_train[feature_cols].select_dtypes(include=[np.number]).columns.tolist()

# Base predictions tracking
oof_preds_cat = np.zeros((len(df_train), len(targets)))
oof_preds_lgb = np.zeros((len(df_train), len(targets)))
oof_preds_xgb = np.zeros((len(df_train), len(targets)))

test_preds_cat = np.zeros((len(df_test), len(targets)))
test_preds_lgb = np.zeros((len(df_test), len(targets)))
test_preds_xgb = np.zeros((len(df_test), len(targets)))

print("Base 3-Model (CatBoost + LGBM + XGBoost) setup for LOSO evaluation complete.")

In [ ]:
print("🚀 Training SOTA-v21 Base Models in 5-Fold LOSO partitions...")

for target in targets:
    t_global_idx = targets.index(target)
    print(f"\n--- Training for Target: {target} ---")

    y = df_train[target]
    has_label = y.notna()
    valid_indices = np.where(has_label)[0]
    valid_idx_set = set(valid_indices)

    for fold, (train_idx, val_idx) in enumerate(cv_folds):
        tr_idx = np.array([i for i in train_idx if i in valid_idx_set])
        va_idx = np.array([i for i in val_idx if i in valid_idx_set])

        if len(tr_idx) == 0 or len(va_idx) == 0:
            continue

        df_train_fold = df_train.copy()
        df_test_fold = df_test.copy()
        y_train_fold = y.copy()

        # Subject Z-scaling
        z_scaler = LeakFreeSubjectZScaler(id_col=id_col, numeric_cols=numeric_cols)
        z_scaler.fit(df_train_fold.iloc[tr_idx])
        df_train_fold = z_scaler.transform(df_train_fold)
        df_test_fold = z_scaler.transform(df_test_fold)

        # Target Encoding
        t_encoder = LeakFreeSubjectTargetEncoder(id_col=id_col)
        t_encoder.fit(df_train_fold.iloc[tr_idx], y_train_fold.iloc[tr_idx])
        df_train_fold = t_encoder.transform(df_train_fold)
        df_test_fold = t_encoder.transform(df_test_fold)

        # Correlation Filter selection
        stability_scores = {}
        candidate_cols = [c for c in df_train_fold.columns if c.endswith("_subject_zscore")]

        for feat in candidate_cols:
            corr = df_train_fold.iloc[tr_idx][feat].corr(y_train_fold.iloc[tr_idx])
            if pd.notna(corr):
                stability_scores[feat] = abs(corr)

        sorted_feats = sorted(stability_scores.items(), key=lambda x: x[1], reverse=True)
        selected_candidates = [f for f, v in sorted_feats[:50] if v >= 0.05]
        active_cols = selected_candidates + ["subject_target_encoded"]

        X_tr = df_train_fold.iloc[tr_idx][active_cols].fillna(0.5)
        y_tr = y_train_fold.iloc[tr_idx]
        X_va = df_train_fold.iloc[va_idx][active_cols].fillna(0.5)
        X_te = df_test_fold[active_cols].fillna(0.5)

        # 1. CatBoost
        cat_model = CatBoostClassifier(iterations=200, depth=3, learning_rate=0.03, random_state=42, verbose=0)
        cat_model.fit(X_tr, y_tr)
        oof_preds_cat[va_idx, t_global_idx] = cat_model.predict_proba(X_va)[:, 1]
        test_preds_cat[:, t_global_idx] += cat_model.predict_proba(X_te)[:, 1] / n_splits

        # 2. LightGBM
        lgb_model = lgb.LGBMClassifier(n_estimators=100, max_depth=3, learning_rate=0.03, random_state=42, verbose=-1)
        lgb_model.fit(X_tr, y_tr)
        oof_preds_lgb[va_idx, t_global_idx] = lgb_model.predict_proba(X_va)[:, 1]
        test_preds_lgb[:, t_global_idx] += lgb_model.predict_proba(X_te)[:, 1] / n_splits

        # 3. XGBoost
        xgb_model = xgb.XGBClassifier(n_estimators=100, max_depth=3, learning_rate=0.03, random_state=42, eval_metric="logloss")
        xgb_model.fit(X_tr, y_tr)
        oof_preds_xgb[va_idx, t_global_idx] = xgb_model.predict_proba(X_va)[:, 1]
        test_preds_xgb[:, t_global_idx] += xgb_model.predict_proba(X_te)[:, 1] / n_splits

    print(f"Target {target} CV Scores:")
    print(f"  - CatBoost: {log_loss(y[has_label], oof_preds_cat[has_label, t_global_idx]):.5f}")
    print(f"  - LightGBM: {log_loss(y[has_label], oof_preds_lgb[has_label, t_global_idx]):.5f}")
    print(f"  - XGBoost:  {log_loss(y[has_label], oof_preds_xgb[has_label, t_global_idx]):.5f}")

## 📈 Stacking & Blending Decision Sweep (v21 Meta Stacker)

In this block, we optimize model weights using the SLSQP algorithm and train a Logistic Regression meta-learner. The system automatically selects the best option per target.

In [ ]:
# =========================================================================
# 📝 submissions/submission_log.csv 자동 기록(Append) 및 고유 사본 저장 모듈
# =========================================================================
import datetime
import shutil

log_path = PROJECT_ROOT / "submissions" / "submission_log.csv" if 'PROJECT_ROOT' in locals() else Path("submissions/submission_log.csv")

if log_path.exists():
    log_df = pd.read_csv(log_path)
    new_id = log_df['id'].max() + 1
else:
    log_df = pd.DataFrame(columns=['id', 'date', 'submission_file', 'model_description', 'local_cv_score', 'public_lb_score', 'is_submitted', 'notes'])
    new_id = 1

current_time = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
mean_loss = np.mean(stacked_oof_overall)

# 1. Define highly unique file name to prevent future overwriting chaos!
unique_sub_name = f"submission_id_{new_id:03d}_score{mean_loss:.4f}.csv"
unique_sub_path = log_path.parent / unique_sub_name

# 2. Save final predictions directly to unique file name
stack_sub_df.to_csv(unique_sub_path, index=False)
print(f"📦 Saved unique SOTA prediction asset to: {unique_sub_path.name}")

# Also save to common file for safety
shutil.copy2(unique_sub_path, log_path.parent / "submission_calibrated_ensemble.csv")

# 3. Create new record row with unique file name mapped
new_row = {
    'id': new_id,
    'date': current_time,
    'submission_file': unique_sub_name,
    'model_description': f'SOTA-v21 Hybrid Stacking-Blending Calibrated Ensemble (LOSO) - CV {mean_loss:.6f}',
    'local_cv_score': round(mean_loss, 6),
    'public_lb_score': 0.0,
    'is_submitted': True,
    'notes': 'Executed automatically via SOTA legacy history pipeline Jupyter Notebook.'
}

# 4. Append and save
log_df = pd.concat([log_df, pd.DataFrame([new_row])], ignore_index=True)
log_df.to_csv(log_path, index=False)
print(f"📝 Successfully appended execution results to submission_log.csv under ID: {new_id}!")